In [96]:
import pandas as pd
import numpy as np
import us

In [97]:
zillow = pd.read_csv('../Data/Zillow ZIP.csv')
hpi = pd.read_csv('../Data/hpi master.csv')

In [98]:
hpi = hpi[hpi['level'] == 'State']

In [99]:
column_names_to_convert = zillow.columns[9:]

new_column_names = pd.to_datetime(column_names_to_convert)

zillow.columns = list(zillow.columns[:9]) + list(new_column_names)

for column in new_column_names:
    column_year = pd.to_datetime(column).year  
    if not (2017 <= column_year <= 2019):  
        zillow = zillow.drop(columns=[column])  

zillow_years = zillow.columns[9:]


current_year = 2017
year_data = 0 * len(zillow)
count = 0
for column in zillow_years:
    column_year = pd.to_datetime(column).year  
    if column_year == current_year:
        year_data += zillow[column]
        count += 1
        zillow = zillow.drop(columns = [column])
    if column_year != current_year:
        year_avg = year_data/count
        zillow[f'zillow {current_year}'] = year_avg
        year_data = 0 * len(zillow)
        current_year = column_year
        count = 0
        zillow = zillow.drop(columns = [column])

if count > 0:
    year_avg = year_data / count
    zillow[f'zillow {current_year}'] = year_avg
        

In [100]:
hpi = hpi[hpi['yr'].isin([2017, 2018, 2019])]

In [101]:
zillow = zillow.dropna()
hpi = hpi.dropna()
hpi['place_id'] = pd.to_numeric(hpi['place_id'],errors = 'coerce')
hpi = hpi[hpi['hpi_flavor'] == 'expanded-data']

In [104]:
hpi['State'] = hpi['place_name'].apply(lambda x: x.split(', ')[-1])


hpi_states = hpi.groupby(['State', 'yr']).agg({
    'index_sa': 'mean'
}).reset_index()


In [105]:
hpi_pivot = hpi_states.pivot(index='State', columns='yr', values='index_sa')

hpi_pivot = hpi_pivot.reset_index()

In [107]:
state_to_abbr = {state.name: state.abbr for state in us.states.STATES}

# Replace the state names with abbreviations
hpi_pivot['State'] = hpi_pivot['State'].map(state_to_abbr).fillna(hpi_pivot['State'])  

# Step 2: Merge any duplicate rows by grouping by 'State' and averaging the values for each year
hpi_pivot_merged = hpi_pivot.groupby('State').mean().reset_index()

hpi_pivot_merged.columns = [str(col).replace(' ', '') for col in hpi_pivot_merged.columns]


In [108]:
zillow_states = zillow.groupby('State').agg({
    'zillow 2017': 'mean',
    'zillow 2018': 'mean',
    'zillow 2019': 'mean'
}).reset_index()

In [109]:
merged_states = zillow_states.merge(hpi_pivot_merged, how = 'inner')
merged_states.dropna()

,State,zillow 2017,zillow 2018,zillow 2019,2017,2018,2019
0,AK,281045.549100,286992.662286,302661.641424,257.3650,259.7250,267.3500
1,AL,142021.618914,148187.698248,156324.858927,168.7450,176.2650,184.1850
2,AR,133829.737372,138999.612132,144338.535730,226.2825,234.6775,244.7350
3,AZ,239103.303555,256592.754945,274845.507328,271.9225,293.1425,313.9925
4,CA,577832.078140,642896.065006,653197.806655,259.8100,276.6475,287.6475
5,CO,359477.007785,388177.951356,410138.806359,409.2825,442.5200,467.4250
6,CT,308946.580539,314428.333455,315056.066651,165.4875,171.5975,176.0050
7,DE,262915.768598,270499.405478,279134.177613,186.5475,193.7850,202.7250
8,FL,230760.061838,246564.720747,258163.084709,277.6050,297.8975,315.3325
9,GA,166472.470959,179663.898495,191217.552384,200.8300,216.3250,229.8550


In [110]:
merged_states['zillow 2017-2018'] = (merged_states['zillow 2018'] - merged_states['zillow 2017'])/merged_states['zillow 2017']*100
merged_states['zillow 2018-2019'] = (merged_states['zillow 2019'] - merged_states['zillow 2018'])/merged_states['zillow 2018']*100
merged_states['zillow 2017-2019'] = (merged_states['zillow 2019'] - merged_states['zillow 2017'])/merged_states['zillow 2017']*100

merged_states['hpi 2017-2018'] = (merged_states['2018'] - merged_states['2017'])/merged_states['2017']*100
merged_states['hpi 2018-2019'] = (merged_states['2019'] - merged_states['2018'])/merged_states['2018']*100
merged_states['hpi 2017-2019'] = (merged_states['2019'] - merged_states['2017'])/merged_states['2017']*100

merged_states

,State,zillow 2017,zillow 2018,zillow 2019,2017,2018,2019,zillow 2017-2018,zillow 2018-2019,zillow 2017-2019,hpi 2017-2018,hpi 2018-2019,hpi 2017-2019
0,AK,281045.549100,286992.662286,302661.641424,257.3650,259.7250,267.3500,2.116067,5.459714,7.691313,0.916986,2.935797,3.879704
1,AL,142021.618914,148187.698248,156324.858927,168.7450,176.2650,184.1850,4.341648,5.491118,10.071171,4.456428,4.493235,9.149901
2,AR,133829.737372,138999.612132,144338.535730,226.2825,234.6775,244.7350,3.863024,3.840963,7.852364,3.709964,4.285669,8.154630
3,AZ,239103.303555,256592.754945,274845.507328,271.9225,293.1425,313.9925,7.314600,7.113510,14.948436,7.803694,7.112582,15.471320
4,CA,577832.078140,642896.065006,653197.806655,259.8100,276.6475,287.6475,11.260016,1.602396,13.042843,6.480697,3.976179,10.714561
5,CO,359477.007785,388177.951356,410138.806359,409.2825,442.5200,467.4250,7.984083,5.657419,14.093196,8.120919,5.627994,14.205958
6,CT,308946.580539,314428.333455,315056.066651,165.4875,171.5975,176.0050,1.774337,0.199643,1.977522,3.692122,2.568511,6.355465
7,DE,262915.768598,270499.405478,279134.177613,186.5475,193.7850,202.7250,2.884436,3.192159,6.168671,3.879709,4.613360,8.672054
8,FL,230760.061838,246564.720747,258163.084709,277.6050,297.8975,315.3325,6.848958,4.703984,11.875115,7.309847,5.852684,13.590353
9,GA,166472.470959,179663.898495,191217.552384,200.8300,216.3250,229.8550,7.924089,6.430704,14.864368,7.715481,6.254478,14.452522


In [111]:
tz_2017 = np.mean(merged_states['zillow 2017'])
tz_2018 = np.mean(merged_states['zillow 2018'])
tz_2019 = np.mean(merged_states['zillow 2019'])
thpi_2017 = np.mean(merged_states['2017'])
thpi_2018 = np.mean(merged_states['2018'])
thpi_2019 = np.mean(merged_states['2019'])

#percent changes for US
print('Percent Changes US')
print(f'zillow 2017-2018 {(tz_2018 - tz_2017)/tz_2017*100}')
print(f'zillow 2018-2019 {(tz_2019 - tz_2018)/tz_2018*100}')
print(f'zillow 2017-2019 {(tz_2019 - tz_2017)/tz_2017*100}')
print(f'hpi 2017-2018 {(thpi_2018 - thpi_2017)/thpi_2017*100}')
print(f'hpi 2018-2019 {(thpi_2019 - thpi_2018)/thpi_2018*100}')
print(f'hpi 2017-2019 {(thpi_2019 - thpi_2017)/thpi_2019*100}')

Percent Changes US
zillow 2017-2018 5.999550248001091
zillow 2018-2019 4.332477596629275
zillow 2017-2019 10.591957015023528
hpi 2017-2018 5.897908445657871
hpi 2018-2019 5.305743228696727
hpi 2017-2019 10.327235285170353
